# Dermatology arm, step 3: the locked analysis (H1 to H6)

Follows `data/derm/derm_lock_v1.json`; external sets are analysed once here. Referral: each model refers its top f of images, f = 1.31 x malignant prevalence (secondary: f = prevalence). Scores: malignancy score (summed MEL+BCC+SCC probability). Intervals: patient-cluster bootstrap (1,000) unless stated. Plus a descriptive label-free sibling check (agreement by relationship). CPU, High-RAM; about 20 to 40 minutes.

In [1]:
# Cell 1: load
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr, norm, multivariate_normal
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score
DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'; PR, FP = f'{DERM}/predictions', f'{DERM}/fingerprints'
FIG = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/05_figures'
CLASSES = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
ARCHS = ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2']; SOURCES = ['HAM10000', 'BCN20000']; SEEDS = [0, 1]
MODELS = [(s, a, k) for s in SOURCES for a in ARCHS for k in SEEDS]; mname = lambda s, a, k: f'{s}_{a}_seed{k}'
VARIANTS = [(s, a) for s in SOURCES for a in ARCHS]
def clean(s):
    df = pd.read_csv(f'{DERM}/manifest_{s}.csv', dtype={'image_id': str, 'lesion_id': str, 'patient_id': str}).sort_values('image_id')
    return df[~df.duplicated('raw_sha256', keep='first') & df.dx.isin(CLASSES)].reset_index(drop=True)
SETS = {}
for es in ('HIBA', 'PAD'):
    D = clean(es); P = {}
    for s, a, k in MODELS:
        p = pd.read_csv(f'{PR}/{es}__{mname(s, a, k)}.csv', dtype={'image_id': str}).set_index('image_id').reindex(D.image_id)
        assert p.p_malignant.notna().all(); P[(s, a, k)] = p.p_malignant.to_numpy()
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    SETS[es] = {'D': D, 'P': P, 'mal': mal, 'mel': D.dx.eq('MEL').to_numpy(), 'dx': D.dx.to_numpy(),
                'cl': pd.factorize(D.patient_id.astype(str))[0], 'f': 1.31 * mal.mean(), 'f2': mal.mean()}
    print(es, len(D), 'images |', D.patient_id.nunique(), 'patients | malignant', int(mal.sum()), '| melanoma', int(SETS[es]['mel'].sum()), '| f =', round(SETS[es]['f'], 3))
IR = json.load(open(f'{DERM}/inference_record.json')); print('checkpoints hashed at', IR['hashed_utc'])

Mounted at /content/drive
HIBA 1616 images | 623 patients | malignant 751 | melanoma 253 | f = 0.609
PAD 2283 images | 1373 patients | malignant 1081 | melanoma 52 | f = 0.62
checkpoints hashed at 2026-09-27T01:49:27.076383+00:00


In [2]:
# Cell 2: helpers (referral, matched-volume ensemble, per-image vectors averaged over seed combinations, patient bootstrap)
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
RNG = np.random.default_rng(20261603); NB = 1000
def boot_w(cl):
    ncl = cl.max() + 1; return [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[cl].astype(float) for _ in range(NB)]
for es, S in SETS.items():
    n = len(S['mal']); S['W'] = boot_w(S['cl'])
    for key in ('f', 'f2'):
        S[f'R_{key}'] = {m: refer(S['P'][m], S[key]) for m in MODELS}
    S['RK'] = {m: rankdata(S['P'][m]) / n for m in MODELS}
def ci(vals): return [round(float(np.nanpercentile(vals, 2.5)), 4), round(float(np.nanpercentile(vals, 97.5)), 4)]
def combos(va, vb):
    if va == vb: return [((*va, i), (*vb, j)) for i in SEEDS for j in SEEDS if i < j]
    return [((*va, i), (*vb, j)) for i in SEEDS for j in SEEDS]

In [3]:
# Cell 3: H1 benefit and H2 severity gradient (ordered pairs of different architecture; per-image vectors, patient bootstrap)
H1, H2 = {}, {}
for es, S in SETS.items():
    for fk in ('f', 'f2'):
        R = S[f'R_{fk}']; f = S[fk]
        anc = {t: 0 for t in ('mel', 'mal', 'BCC', 'SCC')}; ens = {t: 0 for t in anc}; uni = {t: 0 for t in anc}; cnt = 0
        tg = {'mel': S['mel'], 'mal': S['mal'], 'BCC': S['dx'] == 'BCC', 'SCC': S['dx'] == 'SCC'}
        for va in VARIANTS:
            for vb in VARIANTS:
                if va[1] == vb[1]: continue
                for ka, kb in combos(va, vb):
                    e = refer(np.maximum(S['RK'][ka], S['RK'][kb]), f)
                    for t, y in tg.items():
                        anc[t] = anc[t] + (y & ~R[ka]); ens[t] = ens[t] + (y & ~e); uni[t] = uni[t] + (y & ~R[ka] & R[kb])
                    cnt += 1
        def red(w, t): a_ = anc[t] @ w; return (a_ - ens[t] @ w) / a_ if a_ > 0 else np.nan
        ones = np.ones(len(S['mal'])); out = {}
        for t in anc:
            obs = red(ones, t); bs = [red(w, t) for w in S['W']]
            out[t] = {'relative_reduction_matched_volume': round(float(obs), 4), 'ci95': ci(bs),
                      'deployed_misses_per_pair': round(float(anc[t].sum() / cnt), 2), 'union_rescue_share': round(float(uni[t].sum() / anc[t].sum()), 4) if anc[t].sum() else None}
        d = [red(w, 'mel') - red(w, 'BCC') for w in S['W']]; d2 = [red(w, 'mel') - red(w, 'SCC') for w in S['W']]
        H1[f'{es}_{fk}'] = {k: out[k] for k in ('mel', 'mal')}
        H2[f'{es}_{fk}'] = {'MEL': out['mel'], 'BCC': out['BCC'], 'SCC': out['SCC'],
                            'MEL_minus_BCC': {'mean': round(float(red(ones, 'mel') - red(ones, 'BCC')), 4), 'ci95': ci(d)},
                            'MEL_minus_SCC': {'mean': round(float(red(ones, 'mel') - red(ones, 'SCC')), 4), 'ci95': ci(d2)}}
    print(es, 'H1 (primary f):', json.dumps(H1[f'{es}_f']))
print('H2 HIBA:', json.dumps({k: v for k, v in H2['HIBA_f'].items() if k.startswith('MEL_minus')}))

HIBA H1 (primary f): {"mel": {"relative_reduction_matched_volume": 0.0682, "ci95": [0.0209, 0.1217], "deployed_misses_per_pair": 36.58, "union_rescue_share": 0.5527}, "mal": {"relative_reduction_matched_volume": 0.1671, "ci95": [0.1215, 0.2161], "deployed_misses_per_pair": 71.5, "union_rescue_share": 0.6636}}
PAD H1 (primary f): {"mel": {"relative_reduction_matched_volume": 0.0435, "ci95": [-0.0127, 0.1133], "deployed_misses_per_pair": 26.04, "union_rescue_share": 0.3482}, "mal": {"relative_reduction_matched_volume": 0.0989, "ci95": [0.0744, 0.1236], "deployed_misses_per_pair": 236.33, "union_rescue_share": 0.6227}}
H2 HIBA: {"MEL_minus_BCC": {"mean": -0.2473, "ci95": [-0.3459, -0.1556]}, "MEL_minus_SCC": {"mean": -0.1453, "ci95": [-0.2709, -0.0163]}}


In [4]:
# Cell 4: H3 training data (primary: melanoma co-miss on HIBA, different-architecture pairs, different vs same source)
def ptype(va, vb):
    if va == vb: return 'seed_siblings'
    if va[1] == vb[1]: return 'same_arch_diff_source'
    return 'diff_arch_diff_source' if va[0] != vb[0] else 'diff_arch_same_source'
UNORD = [(va, vb) for i, va in enumerate(VARIANTS) for vb in VARIANTS[i:]]
TYPES = ['seed_siblings', 'same_arch_diff_source', 'diff_arch_same_source', 'diff_arch_diff_source']
H3 = {}
for es, tname in (('HIBA', 'mel'), ('HIBA', 'mal'), ('PAD', 'mal'), ('PAD', 'mel')):
    S = SETS[es]; tgt = S[tname]; R = S['R_f']; VEC = {}
    for va, vb in UNORD:
        cs = combos(va, vb); ma = mb = both = 0
        for ka, kb in cs:
            x, y = tgt & ~R[ka], tgt & ~R[kb]; ma = ma + x; mb = mb + y; both = both + (x & y)
        VEC[(va, vb)] = (ma / len(cs), mb / len(cs), both / len(cs))
    def ratio(p, w):
        ma, mb, both = (x @ w for x in VEC[p]); T = tgt @ w
        return both / (ma * mb / T) if ma * mb > 0 else np.nan
    def summ(w): return {t: float(np.nanmean([ratio(p, w) for p in UNORD if ptype(*p) == t])) for t in TYPES}
    obs = summ(np.ones(len(tgt))); bs = [summ(w) for w in S['W']]
    res = {t: {'mean': round(obs[t], 3), 'ci95': ci([b[t] for b in bs])} for t in TYPES}
    for a_, b_ in (('diff_arch_same_source', 'diff_arch_diff_source'), ('seed_siblings', 'same_arch_diff_source'), ('same_arch_diff_source', 'diff_arch_same_source')):
        res[f'{a_}__minus__{b_}'] = {'mean': round(obs[a_] - obs[b_], 3), 'ci95': ci([b[a_] - b[b_] for b in bs])}
    H3[f'{es}_{tname}'] = res
    print(f'\nH3 {es} {tname}'); print(pd.DataFrame(res).T.to_string())
H3_PRIMARY = H3['HIBA_mel']['diff_arch_same_source__minus__diff_arch_diff_source']
H3_SUPPORTED = H3_PRIMARY['ci95'][0] > 0
print('\nH3 PRIMARY (HIBA melanoma, same-source minus different-source, different architectures):', H3_PRIMARY, '| supported:', H3_SUPPORTED)


H3 HIBA mel
                                                      mean               ci95
seed_siblings                                        4.459   [3.8191, 5.3647]
same_arch_diff_source                                3.224   [2.7396, 3.8884]
diff_arch_same_source                                3.375   [2.9189, 3.9397]
diff_arch_diff_source                                 2.93   [2.4897, 3.5234]
diff_arch_same_source__minus__diff_arch_diff_source  0.444    [0.2263, 0.752]
seed_siblings__minus__same_arch_diff_source          1.236    [0.8384, 1.896]
same_arch_diff_source__minus__diff_arch_same_source -0.151  [-0.5057, 0.1766]

H3 HIBA mal
                                                      mean                ci95
seed_siblings                                        5.911    [5.3573, 6.6886]
same_arch_diff_source                                3.505    [2.9999, 4.0457]
diff_arch_same_source                                4.091    [3.6884, 4.5646]
diff_arch_diff_source             

In [5]:
# Cell 5: H4 variance decomposition (latent correlation of malignancy misses on HIBA, all 276 model pairs)
def tetra(p11, pa, pb):
    if min(pa, pb) <= 0 or max(pa, pb) >= 1: return np.nan
    za, zb = norm.ppf(pa), norm.ppf(pb)
    try: return brentq(lambda r: multivariate_normal.cdf([za, zb], mean=[0, 0], cov=[[1, r], [r, 1]]) - p11, -0.999, 0.999)
    except ValueError: return np.nan
S = SETS['HIBA']; tgt = S['mal']; R = S['R_f']; T = tgt.sum(); rows = []
for ma_, mb_ in itertools.combinations(MODELS, 2):
    x, y = tgt & ~R[ma_], tgt & ~R[mb_]
    rows.append({'rho': tetra((x & y).sum() / T, x.sum() / T, y.sum() / T), 'same_source': int(ma_[0] == mb_[0]), 'same_arch': int(ma_[1] == mb_[1]),
                 'seed_sibling': int(ma_[:2] == mb_[:2])})
V = pd.DataFrame(rows).dropna(); X = np.column_stack([np.ones(len(V)), V.same_source, V.same_arch, V.seed_sibling])
beta, *_ = np.linalg.lstsq(X, V.rho.values, rcond=None)
H4 = {'floor_intercept': round(float(beta[0]), 4), 'same_training_source': round(float(beta[1]), 4), 'same_architecture_checkpoint': round(float(beta[2]), 4),
      'seed_sibling_extra': round(float(beta[3]), 4), 'n_pairs': int(len(V)), 'outcome': 'latent correlation of malignancy misses, HIBA, f = 1.31 x prevalence'}
print('H4:', H4)

H4: {'floor_intercept': 0.4681, 'same_training_source': 0.1287, 'same_architecture_checkpoint': 0.0392, 'seed_sibling_extra': 0.1437, 'n_pairs': 276, 'outcome': 'latent correlation of malignancy misses, HIBA, f = 1.31 x prevalence'}


In [6]:
# Cell 6: H5 degradation fingerprint (measured on HIBA half A, unlabelled; shared misses scored on half B), partial rank correlation given strength
half_a = set(pd.read_csv(f'{FP}/hiba_half_A_patients.csv', dtype={'patient_id': str}).patient_id)
S = SETS['HIBA']; B = ~S['D'].patient_id.astype(str).isin(half_a).to_numpy(); tgt = S['mal'] & B; R = S['R_f']
FPX = {m: dict(np.load(f'{FP}/{mname(*m)}.npz', allow_pickle=True)) for m in MODELS}
GROUPS = {'degradation': ['jpeg_q15', 'gamma_dark', 'gamma_bright', 'noise', 'vignette', 'blur_r3'], 'frequency': ['lowpass_r2', 'lowpass_r4', 'highpass_r4'], 'channel': ['drop_red', 'drop_green', 'drop_blue']}
MISS = {m: int((tgt & ~R[m]).sum()) for m in MODELS}; rows = []
for ma_, mb_ in itertools.combinations(MODELS, 2):
    x, y = tgt & ~R[ma_], tgt & ~R[mb_]; Tn = tgt.sum()
    r = {'a': ma_, 'b': mb_, 'comiss': ((x & y).sum() / Tn) / ((x.sum() / Tn) * (y.sum() / Tn)) if x.sum() and y.sum() else np.nan,
         'mean_miss': (MISS[ma_] + MISS[mb_]) / 2, 'gap_miss': abs(MISS[ma_] - MISS[mb_])}
    for g, ks in GROUPS.items(): r[f'sim_{g}'] = float(np.mean([spearmanr(FPX[ma_][f'resp_{k}'], FPX[mb_][f'resp_{k}'])[0] for k in ks]))
    rows.append(r)
F5 = pd.DataFrame(rows).dropna(subset=['comiss'])
def partial(df, p):
    Z_ = np.column_stack([np.ones(len(df)), rankdata(df.mean_miss), rankdata(df.gap_miss)])
    rx = rankdata(df[p]) - Z_ @ np.linalg.lstsq(Z_, rankdata(df[p]), rcond=None)[0]; ry = rankdata(df.comiss) - Z_ @ np.linalg.lstsq(Z_, rankdata(df.comiss), rcond=None)[0]
    return float(np.corrcoef(rx, ry)[0, 1])
H5 = {}
for g in GROUPS:
    obs = partial(F5, f'sim_{g}'); bs = []
    for _ in range(1000):
        keep = set(map(tuple, np.array(MODELS, dtype=object)[RNG.choice(len(MODELS), len(MODELS), replace=True)].tolist()))
        sub = F5[F5.a.isin(keep) & F5.b.isin(keep)]
        if len(sub) > 20: bs.append(partial(sub, f'sim_{g}'))
    H5[g] = {'partial_rho_given_strength': round(obs, 3), 'model_bootstrap_ci95': ci(bs), 'raw_rho': round(float(spearmanr(F5[f'sim_{g}'], F5.comiss)[0]), 3)}
H5_SUPPORTED = H5['degradation']['model_bootstrap_ci95'][0] > 0
print('H5:', json.dumps(H5), '| degradation supported:', H5_SUPPORTED)

H5: {"degradation": {"partial_rho_given_strength": 0.512, "model_bootstrap_ci95": [0.3696, 0.6347], "raw_rho": 0.526}, "frequency": {"partial_rho_given_strength": 0.487, "model_bootstrap_ci95": [0.3654, 0.6217], "raw_rho": 0.503}, "channel": {"partial_rho_given_strength": 0.492, "model_bootstrap_ci95": [0.3356, 0.6553], "raw_rho": 0.512}} | degradation supported: True


In [7]:
# Cell 7: H6 enriched partner choice on HIBA (k melanoma + k non-melanoma; partner of a different architecture), and label-free sibling check
S = SETS['HIBA']; sev = S['mel']; R = S['R_f']; n = len(sev); M = [m for m in MODELS if m[2] == 0]      # one seed per variant, as a site would see one product each
elig = {a: [b for b in M if b[1] != a[1]] for a in M}
net = {(a, b): (sev & ~R[a]).astype(np.int8) - (sev & ~refer(np.maximum(S['RK'][a], S['RK'][b]), S['f'])).astype(np.int8) for a in M for b in elig[a]}
ts = {a: min(elig[a], key=lambda b: int((sev & ~R[b]).sum())) for a in M}; H6 = {}
si, ni = np.flatnonzero(sev), np.flatnonzero(~sev)
for k in (25, 50):
    out = {r: [] for r in ('enriched_auc', 'random', 'true_strongest', 'oracle')}
    for _ in range(200):
        m = np.zeros(n, bool); m[RNG.choice(si, k, replace=False)] = True; m[RNG.choice(ni, k, replace=False)] = True; ev = ~m
        auc = {b: roc_auc_score(sev[m], S['P'][b][m]) for b in M}; v = {r: [] for r in out}
        for a in M:
            sc = {b: int(net[(a, b)][ev].sum()) for b in elig[a]}
            v['random'].append(np.mean(list(sc.values()))); v['oracle'].append(max(sc.values())); v['true_strongest'].append(sc[ts[a]]); v['enriched_auc'].append(sc[max(elig[a], key=lambda b: auc[b])])
        for r in out: out[r].append(np.mean(v[r]))
    rnd, tsv = np.mean(out['random']), np.mean(out['true_strongest'])
    H6[k] = {'share_of_true_strongest': round(float((np.mean(out['enriched_auc']) - rnd) / (tsv - rnd)), 3) if tsv != rnd else None,
             'mean_net_melanoma_rescued': {r: round(float(np.mean(x)), 2) for r, x in out.items()}}
print('H6:', H6)
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
SIB = []
for es, S in SETS.items():
    for ma_, mb_ in itertools.combinations(MODELS, 2):
        rel = 'seed_siblings' if ma_[:2] == mb_[:2] else ('same_arch_diff_source' if ma_[1] == mb_[1] else ('diff_arch_same_source' if ma_[0] == mb_[0] else 'diff_arch_diff_source'))
        SIB.append({'set': es, 'relationship': rel, 'kappa': kappa(S['R_f'][ma_], S['R_f'][mb_]), 'rank_corr': float(np.corrcoef(S['RK'][ma_], S['RK'][mb_])[0, 1])})
SIB = pd.DataFrame(SIB); SIB_SUM = SIB.groupby(['set', 'relationship'])[['kappa', 'rank_corr']].median().round(3)
print(SIB_SUM.to_string())

H6: {25: {'share_of_true_strongest': 0.284, 'mean_net_melanoma_rescued': {'enriched_auc': 4.03, 'random': 3.37, 'true_strongest': 5.69, 'oracle': 9.18}}, 50: {'share_of_true_strongest': 0.404, 'mean_net_melanoma_rescued': {'enriched_auc': 3.88, 'random': 3.05, 'true_strongest': 5.11, 'oracle': 8.4}}}
                            kappa  rank_corr
set  relationship                           
HIBA diff_arch_diff_source  0.565      0.705
     diff_arch_same_source  0.626      0.780
     same_arch_diff_source  0.584      0.725
     seed_siblings          0.706      0.863
PAD  diff_arch_diff_source  0.263      0.387
     diff_arch_same_source  0.373      0.533
     same_arch_diff_source  0.283      0.418
     seed_siblings          0.541      0.714


In [8]:
# Cell 8: write results and figure
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
lab = {'seed_siblings': 'seed siblings', 'same_arch_diff_source': 'same arch,\ndiff data', 'diff_arch_same_source': 'diff arch,\nsame data', 'diff_arch_diff_source': 'diff arch,\ndiff data'}
for ax, key, title in zip(axes, ('HIBA_mel', 'HIBA_mal', 'PAD_mal'), ('a  HIBA melanoma (primary)', 'b  HIBA any malignancy', 'c  PAD-UFES-20 any malignancy')):
    r = H3[key]; y = [r[t]['mean'] for t in TYPES]; lo = [r[t]['mean'] - r[t]['ci95'][0] for t in TYPES]; hi = [r[t]['ci95'][1] - r[t]['mean'] for t in TYPES]
    ax.bar(range(4), y, yerr=[lo, hi], color=['#D55E00', '#E69F00', '#56B4E9', '#0072B2'], capsize=4); ax.set_xticks(range(4)); ax.set_xticklabels([lab[t] for t in TYPES], fontsize=8)
    ax.axhline(1, color='#333', lw=0.8, ls=':'); ax.set_title(title, loc='left', fontsize=10, weight='bold'); ax.set_ylabel('Co-miss ratio (1 = independent)'); ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Dermatology: shared blind spots by relationship between the two models (patient-bootstrap 95% intervals)', x=0.01, ha='left', fontsize=11, weight='bold')
fig.tight_layout(); fig.savefig(f'{FIG}/derm_h3_comiss_v2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'lock': 'derm_lock_v1.json', 'checkpoints_hashed_utc': IR['hashed_utc'],
       'sets': {es: {'images': int(len(S['mal'])), 'patients': int(S['cl'].max() + 1), 'malignant': int(S['mal'].sum()), 'melanoma': int(S['mel'].sum()), 'f': round(S['f'], 4)} for es, S in SETS.items()},
       'H1_benefit': H1, 'H2_severity_gradient': H2, 'H3_training_data': H3, 'H3_primary': H3_PRIMARY, 'H3_supported': bool(H3_SUPPORTED),
       'H4_variance_decomposition': H4, 'H5_fingerprint': H5, 'H5_supported': bool(H5_SUPPORTED), 'H6_enriched_partner_choice': H6,
       'label_free_agreement_by_relationship': {f'{i[0]}|{i[1]}': v for i, v in SIB_SUM.to_dict(orient='index').items()}}
json.dump(res, open(f'{DERM}/derm_results_v1.json', 'w'), indent=2, default=str)
print('Wrote data/derm/derm_results_v1.json and 05_figures/derm_h3_comiss_v2.png')

Wrote data/derm/derm_results_v1.json and 05_figures/derm_h3_comiss_v2.png
